# Stage 1 — SPADL Conversion & Feature Engineering
**This notebook is Stage 1 of 2.** Stage 2 (Elo + final merge) runs separately to keep RAM usage low.

**What this notebook does:**

1. Loads raw Wyscout 2017/18 events & match metadata from `WyScout/` (run `python get_data.py` first).
2. Converts each event to a SPADL action (type, result, bodypart, coordinates) **without** using the `socceraction` library — everything is plain `pandas` / `numpy`.
3. Engineers the features used by the models:
   - Base SPADL features (type / result / bodypart / start & end position)
   - 3-action rolling window (attributes of `a-1` and `a-2`)
   - Geometric features (distance + angle to goal, Δx/Δy, inter-action gaps, possession change)
   - Game-context features (score, goal difference)
   - Late-game features (`minutes_remaining`, `score_state`, `score_margin`, `is_substitute`, `minutes_on_pitch`)

**Memory strategy:** process one league at a time, save a per-league Parquet, then `del` the league's raw events and run Python's garbage collector before loading the next league. Peak RAM stays low.

**Inputs** — `WyScout/`:
- `events/events_<Country>.json`
- `matches/matches_<Country>.json`
- `players.json`, `teams.json`, `competitions.json`

**Outputs** — `intermediate/`:
- `stage1_actions_<Country>.parquet` (one per league)
- `match_meta.parquet`
- `player_windows.parquet`

Stage 2 consumes these files; no CSV / JSON is written here.

## 0. Setup

In [ ]:
!pip install -q tqdm pyarrow

In [ ]:
import json
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
print("pandas:", pd.__version__, "| numpy:", np.__version__)

In [ ]:
# ---- Path configuration ----
# Repository root (the notebooks live one level below it)
PROJECT_ROOT = Path('..') if Path.cwd().name in ('pipeline-stages', 'feature-engineering') else Path('.')
WYSCOUT_ROOT  = PROJECT_ROOT / "WyScout"
INTERIM_DIR   = PROJECT_ROOT / "intermediate"      # Stage 1 outputs live here

EVENTS_DIR    = WYSCOUT_ROOT / "events"
MATCHES_DIR   = WYSCOUT_ROOT / "matches"

LEAGUES = {
    "England": "Premier League",
    "Spain":   "La Liga",
    "Italy":   "Serie A",
    "France":  "Ligue 1",
    "Germany": "Bundesliga",
}

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

assert WYSCOUT_ROOT.exists(), f"Missing {WYSCOUT_ROOT}"
assert EVENTS_DIR.exists(),   f"Missing {EVENTS_DIR}"
assert MATCHES_DIR.exists(),  f"Missing {MATCHES_DIR}"
print("Paths OK. Intermediate outputs will be written to:", INTERIM_DIR)

## 1. Wyscout & SPADL constants

Wyscout (Pappalardo et al., 2019) encodes every event as a combination of `eventId`, `subEventId`, and a list of `tags`. The constants below are the published IDs we need to decode events into SPADL action types.

In [ ]:
# ---- Wyscout primary event IDs ----
E_DUEL, E_FOUL, E_FREE_KICK, E_GK_LEAVING, E_INTERRUPTION = 1, 2, 3, 4, 5
E_OFFSIDE, E_OTHERS, E_PASS, E_SAVE, E_SHOT               = 6, 7, 8, 9, 10

# ---- Wyscout sub-event IDs ----
SE_CROSS         = 80
SE_THROW_IN      = 36
SE_CORNER_SHORT  = 30
SE_CORNER_CROSS  = 31
SE_FK_CROSS      = 32   # Free kick cross  (confirmed subEventId=32)
SE_FK_SHOT       = 33   # Free kick shot   (confirmed subEventId=33)
SE_GOAL_KICK     = 34   # Goal kick        (confirmed subEventId=34)
SE_PENALTY       = 35   # Penalty kick     (confirmed subEventId=35)
SE_ACCELERATION  = 70
SE_CLEARANCE     = 71
SE_TOUCH         = 72
SE_GROUND_ATT    = 11    # ground attacking duel
SE_GROUND_DEF    = 12    # ground defending duel
SE_GROUND_LOOSE  = 13    # ground loose-ball duel
SE_AIR_DUEL      = 10

# ---- Wyscout tag IDs ----
TAG_GOAL          = 101
TAG_OWN_GOAL      = 102
TAG_LEFT_FOOT     = 401
TAG_RIGHT_FOOT    = 402
TAG_HEAD_BODY     = 403
TAG_LOST          = 701
TAG_WON           = 703
TAG_ACCURATE      = 1801
TAG_NOT_ACCURATE  = 1802
TAG_RED_CARD      = 1701
TAG_YELLOW_CARD   = 1702
TAG_SECOND_YELLOW = 1703

# ---- SPADL vocabulary (Decroos et al., 2019) ----
SPADL_ACTION_TYPES = [
    "pass", "cross", "throw_in",
    "freekick_crossed", "freekick_short",
    "corner_crossed",   "corner_short",
    "take_on", "foul", "tackle", "interception",
    "shot", "shot_penalty", "shot_freekick", "goalkick",
    "keeper_save", "keeper_claim", "keeper_punch", "keeper_pick_up",
    "clearance", "bad_touch", "non_action",
]
ACTION_TYPE_TO_ID = {n: i for i, n in enumerate(SPADL_ACTION_TYPES)}

SPADL_RESULTS = ["fail", "success", "offside", "owngoal", "yellow_card", "red_card"]
RESULT_TO_ID  = {n: i for i, n in enumerate(SPADL_RESULTS)}

SPADL_BODYPARTS = ["foot", "head", "other"]
BODYPART_TO_ID  = {n: i for i, n in enumerate(SPADL_BODYPARTS)}

# ---- Pitch dimensions ----
FIELD_LENGTH = 105.0
FIELD_WIDTH  = 68.0
GOAL_WIDTH   = 7.32
GOAL_X, GOAL_Y = FIELD_LENGTH, FIELD_WIDTH / 2.0   # (105, 34)

## 2. Helper functions

- `load_json` — read a Wyscout JSON file.
- `tags_to_set` — convert the `tags` list to a `frozenset` of int IDs (fast `in` checks).
- `extract_positions` — unpack the 1-or-2-element `positions` array into four coords.
- `decode_action` — map `(eventId, subEventId, tags)` to `(action_type, result, bodypart)`.

In [ ]:
def load_json(path: Path) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return pd.DataFrame(data)


def _find_lookup(name: str) -> Path:
    for c in [WYSCOUT_ROOT / f"{name}.json",
              WYSCOUT_ROOT / "lookups" / f"{name}.json",
              WYSCOUT_ROOT / name / f"{name}.json"]:
        if c.exists():
            return c
    raise FileNotFoundError(f"Could not find {name}.json under {WYSCOUT_ROOT}")


def tags_to_set(tags_field):
    if not isinstance(tags_field, list):
        return frozenset()
    out = set()
    for t in tags_field:
        if isinstance(t, dict) and "id" in t:
            out.add(int(t["id"]))
    return frozenset(out)


def extract_positions(pos_field):
    """Return (sx, sy, ex, ey) in the raw 0-100 grid. Single-position events
    (shots w/o defined end) re-use the start as the end."""
    if not isinstance(pos_field, list) or len(pos_field) == 0:
        return np.nan, np.nan, np.nan, np.nan
    s = pos_field[0]
    e = pos_field[1] if len(pos_field) > 1 else pos_field[0]
    sx = s.get("x") if isinstance(s, dict) else np.nan
    sy = s.get("y") if isinstance(s, dict) else np.nan
    ex = e.get("x") if isinstance(e, dict) else np.nan
    ey = e.get("y") if isinstance(e, dict) else np.nan
    return sx, sy, ex, ey

In [ ]:
def decode_action(event_id: int, sub_event_id: int, tag_set: frozenset):
    """Map a raw Wyscout event to SPADL (action_type, result, bodypart)."""

    # ---- Body part ----
    if TAG_HEAD_BODY in tag_set:
        bodypart = "head"
    elif TAG_LEFT_FOOT in tag_set or TAG_RIGHT_FOOT in tag_set:
        bodypart = "foot"
    else:
        bodypart = "foot"   # SPADL default

    # ---- Result ----
    if event_id == E_SHOT or sub_event_id == SE_FK_SHOT or sub_event_id == SE_PENALTY:
        if TAG_GOAL in tag_set:        result = "success"
        elif TAG_OWN_GOAL in tag_set:  result = "owngoal"
        else:                          result = "fail"
    elif event_id == E_FOUL:
        if TAG_RED_CARD in tag_set or TAG_SECOND_YELLOW in tag_set:
            result = "red_card"
        elif TAG_YELLOW_CARD in tag_set:
            result = "yellow_card"
        else:
            result = "success"
    elif event_id == E_DUEL:
        if TAG_WON in tag_set:    result = "success"
        elif TAG_LOST in tag_set: result = "fail"
        else:                     result = "fail"
    else:
        if TAG_ACCURATE in tag_set:       result = "success"
        elif TAG_NOT_ACCURATE in tag_set: result = "fail"
        else:                             result = "success"

    # ---- Action type ----
    if event_id == E_SHOT:
        action_type = "shot"
    elif event_id == E_FREE_KICK and sub_event_id == SE_PENALTY:    # 35 = penalty kick
        action_type = "shot_penalty"
    elif event_id == E_FREE_KICK and sub_event_id == SE_FK_SHOT:    # 33 = free kick shot
        action_type = "shot_freekick"
    elif event_id == E_FREE_KICK and sub_event_id == SE_GOAL_KICK:  # 34 = goal kick
        action_type = "goalkick"
    elif event_id == E_FREE_KICK and sub_event_id == SE_FK_CROSS:   # 32 = free kick cross
        action_type = "freekick_crossed"
    elif event_id == E_FREE_KICK and sub_event_id == SE_THROW_IN:
        action_type = "throw_in"
        bodypart = "other"
    elif event_id == E_FREE_KICK and sub_event_id == SE_CORNER_CROSS:
        action_type = "corner_crossed"
    elif event_id == E_FREE_KICK and sub_event_id == SE_CORNER_SHORT:
        action_type = "corner_short"
    elif event_id == E_FREE_KICK:
        action_type = "freekick_short"
    elif event_id == E_PASS and sub_event_id == SE_CROSS:
        action_type = "cross"
    elif event_id == E_PASS:
        action_type = "pass"
    elif event_id == E_DUEL and sub_event_id == SE_GROUND_ATT:
        action_type = "take_on"
    elif event_id == E_DUEL and sub_event_id == SE_GROUND_DEF:
        action_type = "tackle"
    elif event_id == E_DUEL and sub_event_id == SE_GROUND_LOOSE:
        action_type = "interception" if result == "success" else "tackle"
    elif event_id == E_DUEL:
        action_type = "take_on"
    elif event_id == E_FOUL:
        action_type = "foul"
    elif event_id == E_SAVE:
        action_type = "keeper_save"
    elif event_id == E_OTHERS and sub_event_id == SE_CLEARANCE:
        action_type = "clearance"
    elif event_id == E_OTHERS and sub_event_id == SE_TOUCH:
        action_type = "bad_touch" if result == "fail" else "pass"
    elif event_id == E_OTHERS and sub_event_id == SE_ACCELERATION:
        action_type = "take_on"
    else:
        action_type = "non_action"

    return action_type, result, bodypart

## 3. Load lookups and match metadata (all leagues)

Lookups (`players`, `teams`, `competitions`) and match metadata are small — a few MB each — so we can load them for all leagues upfront. The **raw events** are the huge ones; those are handled league-by-league in §5.

In [ ]:
players_df      = load_json(_find_lookup("players"))
teams_df        = load_json(_find_lookup("teams"))
competitions_df = load_json(_find_lookup("competitions"))

print(f"players:      {len(players_df):,} rows")
print(f"teams:        {len(teams_df):,} rows")
print(f"competitions: {len(competitions_df):,} rows")

In [ ]:
matches_by_league = {}
for country in LEAGUES:
    df = load_json(MATCHES_DIR / f"matches_{country}.json")
    df["league_country"] = country
    df["league_name"]    = LEAGUES[country]
    matches_by_league[country] = df
    print(f"{country:8s}: {len(df):4d} matches")

matches_all = pd.concat(matches_by_league.values(), ignore_index=True)
print(f"\nTotal matches: {len(matches_all):,}")

## 4. Match metadata & player lineup windows

These are small tables (< 50k rows each) computed once from `matchesData.teamsData`. Both get saved as Parquet for Stage 2.

- **`match_meta`** — one row per match: dates, home/away IDs, final score, gameweek.
- **`player_windows`** — one row per (game, player): entry minute, exit minute, starter flag. Used later to compute `is_substitute` and `minutes_on_pitch`.

In [ ]:
def parse_match_meta(matches_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, m in matches_df.iterrows():
        td = m["teamsData"]
        home_tid = away_tid = None
        home_score = away_score = None
        for tid, info in td.items():
            if info.get("side") == "home":
                home_tid, home_score = int(tid), int(info.get("score", 0))
            elif info.get("side") == "away":
                away_tid, away_score = int(tid), int(info.get("score", 0))
        rows.append({
            "game_id":        int(m["wyId"]),
            "league_country": m["league_country"],
            "league_name":    m["league_name"],
            "dateutc":        pd.to_datetime(m["dateutc"], errors="coerce"),
            "gameweek":       int(m.get("gameweek", 0) or 0),  # 0 = unknown; exclude from burn-in check in Stage 2
            "home_team_id":   home_tid,
            "away_team_id":   away_tid,
            "home_score":     home_score,
            "away_score":     away_score,
        })
    return pd.DataFrame(rows)


match_meta = parse_match_meta(matches_all)
print(f"Match metadata: {len(match_meta):,} rows")
match_meta.head(3)

In [ ]:
def extract_player_windows(matches_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, m in matches_df.iterrows():
        game_id = int(m["wyId"])
        for tid, info in m["teamsData"].items():
            tid = int(tid)
            formation = info.get("formation", {}) or {}
            lineup = formation.get("lineup", []) or []
            bench  = formation.get("bench",  []) or []
            subs   = formation.get("substitutions", []) or []
            if not isinstance(subs, list):
                subs = []
            subs_in  = {int(s["playerIn"]):  int(s["minute"])
                        for s in subs if isinstance(s, dict)}
            subs_out = {int(s["playerOut"]): int(s["minute"])
                        for s in subs if isinstance(s, dict)}
            # Starters: entry 0, exit = subs_out if subbed off, else 90.
            for p in lineup:
                pid = int(p["playerId"])
                rows.append({"game_id": game_id, "team_id": tid, "player_id": pid,
                             "is_starter": True,  "entry_minute": 0,
                             "exit_minute": subs_out.get(pid, 90)})
            # Bench players who actually came on.
            for p in bench:
                pid = int(p["playerId"])
                if pid in subs_in:
                    rows.append({"game_id": game_id, "team_id": tid, "player_id": pid,
                                 "is_starter": False, "entry_minute": subs_in[pid],
                                 "exit_minute": subs_out.get(pid, 90)})
    return pd.DataFrame(rows)


player_windows = extract_player_windows(matches_all)
print(f"Player windows: {len(player_windows):,} rows")
player_windows.head(3)

In [ ]:
# Save both tables as Parquet for Stage 2 to read.
match_meta.to_parquet(INTERIM_DIR / "match_meta.parquet", index=False)
player_windows.to_parquet(INTERIM_DIR / "player_windows.parquet", index=False)
print("Saved:")
print(" ", INTERIM_DIR / "match_meta.parquet")
print(" ", INTERIM_DIR / "player_windows.parquet")

## 5. SPADL conversion + feature-engineering functions

Each function operates on a **single league's** DataFrame. The driver loop in §6 calls them in sequence and then frees the intermediate DataFrames before moving on to the next league.

### 5.1 Wyscout → SPADL conversion

Steps:
1. Unpack `positions` → `start_x0, start_y0, end_x0, end_y0` (raw 0-100).
2. Convert `tags` → frozenset of int IDs.
3. Decode `(eventId, subEventId, tags)` → `(action_type, result, bodypart)`.
4. Compute `period_id` and `time_seconds`.
5. **Direction normalisation (play-left-to-right):** flip the away team's coordinates (`x → 100 − x`, `y → 100 − y`) so every team attacks 0 → 100 from its own perspective.
6. Scale `(0-100)` → `(0-105, 0-68)` metres.
7. Drop `non_action` rows (offside whistles, ball-out-of-play, etc.) — they're not playable actions and would contaminate Stage 3's VAEP training signal.

In [ ]:
PERIOD_MAP = {"1H": 1, "2H": 2, "E1": 3, "E2": 4, "P": 5}


def convert_to_spadl(country: str,
                     events_df: pd.DataFrame,
                     matches_df: pd.DataFrame) -> pd.DataFrame:
    home_team_by_game = {}
    for _, m in matches_df.iterrows():
        for tid, info in m["teamsData"].items():
            if info.get("side") == "home":
                home_team_by_game[int(m["wyId"])] = int(tid)
                break

    df = events_df.copy()

    df["eventId"]    = pd.to_numeric(df["eventId"],    errors="coerce").fillna(0).astype(int)
    df["subEventId"] = pd.to_numeric(df["subEventId"], errors="coerce").fillna(0).astype(int)

    # 1. Positions
    pos = df["positions"].apply(extract_positions)
    df["start_x0"] = [p[0] for p in pos]
    df["start_y0"] = [p[1] for p in pos]
    df["end_x0"]   = [p[2] for p in pos]
    df["end_y0"]   = [p[3] for p in pos]
    df = df.dropna(subset=["start_x0", "start_y0"]).copy()

    # 2. Tags
    df["tag_set"] = df["tags"].apply(tags_to_set)

    # 3. Decode
    decoded = [
        decode_action(eid, sid, ts)
        for eid, sid, ts in zip(df["eventId"], df["subEventId"], df["tag_set"])
    ]
    df["action_type"] = [d[0] for d in decoded]
    df["result"]      = [d[1] for d in decoded]
    df["bodypart"]    = [d[2] for d in decoded]

    # 4. Period / time
    df["period_id"]    = df["matchPeriod"].map(PERIOD_MAP).astype("Int64")
    df["time_seconds"] = df["eventSec"].astype(float)

    # 5. Direction normalisation
    game_home = df["matchId"].map(home_team_by_game)
    is_away = df["teamId"].astype(int) != game_home.astype("Int64")
    # don't flip if home team is unknown
    is_away = is_away & game_home.notna()
    df["start_x_raw"] = np.where(is_away, 100.0 - df["start_x0"], df["start_x0"])
    df["start_y_raw"] = np.where(is_away, 100.0 - df["start_y0"], df["start_y0"])
    df["end_x_raw"]   = np.where(is_away, 100.0 - df["end_x0"],   df["end_x0"])
    df["end_y_raw"]   = np.where(is_away, 100.0 - df["end_y0"],   df["end_y0"])

    # 6. Scale to metres
    df["start_x"] = df["start_x_raw"] * FIELD_LENGTH / 100.0
    df["start_y"] = df["start_y_raw"] * FIELD_WIDTH  / 100.0
    df["end_x"]   = df["end_x_raw"]   * FIELD_LENGTH / 100.0
    df["end_y"]   = df["end_y_raw"]   * FIELD_WIDTH  / 100.0

    # 7. Rename & integer-encode
    df = df.rename(columns={"matchId": "game_id", "teamId": "team_id",
                            "playerId": "player_id", "id": "event_id"})
    df["league_country"] = country
    df["type_id"]     = df["action_type"].map(ACTION_TYPE_TO_ID).astype("Int64")
    df["result_id"]   = df["result"].map(RESULT_TO_ID).astype("Int64")
    df["bodypart_id"] = df["bodypart"].map(BODYPART_TO_ID).astype("Int64")

    # Drop non-actions (offside, goal kicks, interruptions)
    df = df[df["action_type"] != "non_action"].copy()

    # remove player_id=0 sentinel rows
    n_before = len(df)
    df = df[df["player_id"] != 0].copy()
    n_dropped = n_before - len(df)
    if n_dropped > 0:
        print(f"      Dropped {n_dropped:,} player_id=0 sentinel rows")

    keep = ["game_id", "league_country", "event_id",
            "period_id", "time_seconds",
            "team_id", "player_id",
            "action_type", "type_id", "result", "result_id",
            "bodypart", "bodypart_id",
            "start_x", "start_y", "end_x", "end_y"]
    return df[keep].reset_index(drop=True)

### 5.2 Absolute game-clock minute

In [ ]:
def add_absolute_time(df: pd.DataFrame) -> pd.DataFrame:
    # 1H@0', 2H@45', E1@90', E2@105', penalties@120'.
    offset = {1: 0, 2: 45*60, 3: 90*60, 4: 105*60, 5: 120*60}
    df = df.copy()
    df["absolute_second"] = (df["period_id"].map(offset).fillna(0)
                             + df["time_seconds"])
    df["minute"] = (df["absolute_second"] // 60).astype(int)
    return df

### 5.3 Score-state tracking

Per-game: walk through actions in order, update `(home_goals, away_goals)` on every goal or own-goal, record the score **at the moment each action occurs**.

In [ ]:
SHOT_TYPES = {"shot", "shot_penalty", "shot_freekick"}


def _score_state_one_game(g: pd.DataFrame, home_id: int, away_id: int) -> pd.DataFrame:
    g = g.sort_values(["period_id", "time_seconds"], kind="stable").reset_index(drop=True).copy()
    hg = ag = 0
    hg_b = np.zeros(len(g), dtype=np.int16)
    ag_b = np.zeros(len(g), dtype=np.int16)

    a_type = g["action_type"].astype(str).values
    a_res  = g["result"].astype(str).values
    teams  = g["team_id"].astype(int).values

    for i in range(len(g)):
        hg_b[i], ag_b[i] = hg, ag
        if a_type[i] in SHOT_TYPES and a_res[i] == "success":
            if teams[i] == home_id:   hg += 1
            elif teams[i] == away_id: ag += 1
        elif a_res[i] == "owngoal":
            if teams[i] == home_id:   ag += 1
            elif teams[i] == away_id: hg += 1

    g["home_goals_before"] = hg_b
    g["away_goals_before"] = ag_b
    acting_home = teams == home_id
    g["score_margin"] = np.where(
        acting_home, g["home_goals_before"] - g["away_goals_before"],
        g["away_goals_before"] - g["home_goals_before"])
    g["score_state"] = pd.cut(g["score_margin"],
                              bins=[-np.inf, -0.5, 0.5, np.inf],
                              labels=["chasing", "level", "protecting"]).astype(str)
    g["goals_for"]       = np.where(acting_home, g["home_goals_before"], g["away_goals_before"])
    g["goals_against"]   = np.where(acting_home, g["away_goals_before"], g["home_goals_before"])
    g["goal_difference"] = g["goals_for"] - g["goals_against"]
    return g


def add_score_state(df: pd.DataFrame,
                    home_map: dict, away_map: dict) -> pd.DataFrame:
    parts = []
    for gid, g in df.groupby("game_id", sort=False):
        if gid in home_map:
            parts.append(_score_state_one_game(g, home_map[gid], away_map[gid]))
    return pd.concat(parts, ignore_index=True)

### 5.4 Geometric features

After the direction flip, the attacking goal is always at `(105, 34)`.

In [ ]:
def add_geometric_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["dx"] = df["end_x"] - df["start_x"]
    df["dy"] = df["end_y"] - df["start_y"]
    df["action_distance"]    = np.hypot(df["dx"], df["dy"])
    df["dist_to_goal_start"] = np.hypot(GOAL_X - df["start_x"], GOAL_Y - df["start_y"])
    df["dist_to_goal_end"]   = np.hypot(GOAL_X - df["end_x"],   GOAL_Y - df["end_y"])

    # Visible goal-mouth angle: tan(a) = gw*x / (x^2+y^2-(gw/2)^2)
    for suf, sx, sy in [("start", "start_x", "start_y"),
                        ("end",   "end_x",   "end_y")]:
        x = (GOAL_X - df[sx]).clip(lower=1e-3)
        y = (GOAL_Y - df[sy])
        denom = x**2 + y**2 - (GOAL_WIDTH/2)**2
        df[f"angle_to_goal_{suf}"] = np.arctan2(GOAL_WIDTH * x, denom)
    return df

### 5.5 Three-action rolling window

For each action `a_i` within the same match, attach `a_{i-1}` and `a_{i-2}` fields plus gap / possession-change features.

In [ ]:
ROLL_COLS = ["type_id", "result_id", "bodypart_id",
             "start_x", "start_y", "end_x", "end_y",
             "team_id", "player_id", "absolute_second"]


def add_rolling_window(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["game_id", "period_id", "time_seconds"], kind="stable").copy()
    gb = df.groupby("game_id", sort=False)
    for lag in (1, 2):
        shifted = gb[ROLL_COLS].shift(lag)
        shifted.columns = [f"{c}_a{lag}" for c in shifted.columns]
        df = pd.concat([df, shifted], axis=1)

    df["time_since_prev"]  = df["absolute_second"] - df["absolute_second_a1"]
    df["dx_from_prev"]     = df["start_x"] - df["end_x_a1"]
    df["dy_from_prev"]     = df["start_y"] - df["end_y_a1"]
    df["dist_from_prev"]   = np.hypot(df["dx_from_prev"], df["dy_from_prev"])
    df["possession_change"] = (df["team_id"] != df["team_id_a1"]).astype(int)
    return df

### 5.6 Late-game-specific features

In [ ]:
def add_late_game_features(df: pd.DataFrame,
                            player_windows: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["minutes_remaining"] = (90 - df["minute"]).clip(lower=0)
    df["is_post_75"]        = (df["minute"] >= 75).astype(int)

    # deduplicate player_windows before merging.
    # Wyscout occasionally lists a player in both lineup and bench
    # (data inconsistency), which would silently multiply rows on merge.
    pw = (player_windows
          .drop_duplicates(subset=["game_id", "player_id"], keep="first")
          [["game_id", "player_id", "is_starter", "entry_minute"]])
    df = df.merge(pw, on=["game_id", "player_id"], how="left")

    df["is_substitute"]    = (~df["is_starter"].fillna(True)).astype(int)
    df["entry_minute"]     = df["entry_minute"].fillna(0).astype(int)
    df["minutes_on_pitch"] = (df["minute"] - df["entry_minute"]).clip(lower=0)
    return df

## 6. Per-league driver loop

For each league in turn:

1. Load that league's raw events (~600k–700k rows).
2. Run the SPADL converter.
3. Add absolute time, score state, geometric features, rolling window, late-game features.
4. Save as `stage1_actions_<Country>.parquet`.
5. `del` the intermediate DataFrames and call `gc.collect()` before moving on.

This keeps peak RAM low. You should see memory usage drop back down between leagues.

In [ ]:
home_map = dict(zip(match_meta["game_id"], match_meta["home_team_id"]))
away_map = dict(zip(match_meta["game_id"], match_meta["away_team_id"]))

# Columns to keep in the per-league Parquet (Elo gets added in Stage 2).
STAGE1_COLUMNS = [
    # identifiers
    "game_id", "league_country", "event_id",
    "period_id", "time_seconds", "absolute_second", "minute",
    "team_id", "player_id",
    # base SPADL
    "action_type", "type_id", "result", "result_id",
    "bodypart", "bodypart_id",
    "start_x", "start_y", "end_x", "end_y",
    # geometric
    "dx", "dy", "action_distance",
    "dist_to_goal_start", "dist_to_goal_end",
    "angle_to_goal_start", "angle_to_goal_end",
    # rolling window a-1
    "type_id_a1", "result_id_a1", "bodypart_id_a1",
    "start_x_a1", "start_y_a1", "end_x_a1", "end_y_a1",
    "team_id_a1", "player_id_a1", "absolute_second_a1",
    # rolling window a-2
    "type_id_a2", "result_id_a2", "bodypart_id_a2",
    "start_x_a2", "start_y_a2", "end_x_a2", "end_y_a2",
    "team_id_a2", "player_id_a2", "absolute_second_a2",
    # gaps & possession
    "time_since_prev", "dx_from_prev", "dy_from_prev", "dist_from_prev",
    "possession_change",
    # game context
    "home_goals_before", "away_goals_before",
    "goals_for", "goals_against", "goal_difference",
    "score_margin", "score_state",
    # late-game
    "minutes_remaining", "is_post_75",
    "is_starter", "is_substitute",
    "entry_minute", "minutes_on_pitch",
]

In [ ]:
summary_rows = []

for country in LEAGUES:
    print(f"\n========== {country} ==========")

    # ---- Load raw events for this league only ----
    events_path = EVENTS_DIR / f"events_{country}.json"
    print(f"[1/6] Loading raw events from {events_path.name} ...")
    raw_events = load_json(events_path)
    print(f"      {len(raw_events):,} raw events")

    # ---- SPADL conversion ----
    print("[2/6] Converting to SPADL ...")
    actions = convert_to_spadl(country, raw_events, matches_by_league[country])
    del raw_events
    gc.collect()
    print(f"      {len(actions):,} SPADL actions")

    # ---- Absolute time ----
    print("[3/6] Absolute time + score state ...")
    actions = add_absolute_time(actions)
    actions = add_score_state(actions, home_map, away_map)

    # ---- Geometric + rolling window + late-game ----
    print("[4/6] Geometric + rolling window features ...")
    actions = add_geometric_features(actions)
    actions = add_rolling_window(actions)

    print("[5/6] Late-game features ...")
    actions = add_late_game_features(actions, player_windows)

    # ---- Save as Parquet ----
    keep = [c for c in STAGE1_COLUMNS if c in actions.columns]
    missing = [c for c in STAGE1_COLUMNS if c not in actions.columns]
    if missing:
        print(f"      [warn] missing columns (skipped): {missing}")

    out_path = INTERIM_DIR / f"stage1_actions_{country}.parquet"
    actions[keep].to_parquet(out_path, index=False)
    size_mb = out_path.stat().st_size / 1e6
    print(f"[6/6] Wrote {out_path.name} ({size_mb:.1f} MB, {len(actions):,} rows)")

    summary_rows.append({
        "country":       country,
        "n_actions":     len(actions),
        "n_post75":      int(actions["is_post_75"].sum()),
        "pct_post75":    float(actions["is_post_75"].mean() * 100),
        "file_size_mb":  size_mb,
    })

    # ---- Free memory before next league ----
    del actions
    gc.collect()

print("\n\nDone. Per-league summary:")
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

## 7. Verify outputs

Confirm that all expected files exist.

In [ ]:
expected_files = [INTERIM_DIR / "match_meta.parquet",
                  INTERIM_DIR / "player_windows.parquet"] + \
                 [INTERIM_DIR / f"stage1_actions_{c}.parquet" for c in LEAGUES]

print("Stage 1 intermediate outputs:")
all_ok = True
for f in expected_files:
    if f.exists():
        size_mb = f.stat().st_size / 1e6
        print(f"  [OK]      {f.name:40s}  {size_mb:7.1f} MB")
    else:
        print(f"  [MISSING] {f.name}")
        all_ok = False

if all_ok:
    print("\nAll Stage 1 outputs present. You can now run the Stage 2 notebook.")
else:
    print("\nSome outputs are missing - re-run the failing cell above.")

---

## Done

**Next step:** open the **Stage 2 notebook** (`stage2_elo_and_merge.ipynb`). It reads these intermediate Parquet files, computes Elo ratings, merges them onto every action, and writes the final `data/stage1_stage2_features.csv`.